# Pull Quarterly Earnings Announcement Dates

Pulls Compustat `comp.fundq` field `rdq` (firm-reported quarterly earnings
announcement date), linked to CRSP `permno` via the CRSP-Compustat merged
link table.

Output: one CSV keyed on (`permno`, `announce_date`).

Run on WRDS (JupyterHub or `qrsh`).

In [ ]:
import os
from pathlib import Path
import pandas as pd
import wrds


START = "2018-10-01"
END   = "2024-12-31"
WRDS_USER = os.environ.get("WRDS_USER")


db = wrds.Connection(wrds_username=WRDS_USER) if WRDS_USER else wrds.Connection()
print("connected as", os.environ.get("USER"))

## Compustat quarterly announcement dates

`rdq` is the report date of quarterly earnings as filed by the company.
Joining to `crsp.ccmxpf_linktable` on primary (P) and consolidated (C) links
gives the canonical `permno` for each `gvkey`-quarter pair.

The `indfmt='INDL'`, `datafmt='STD'`, `consol='C'`, `popsrc='D'` filters are
the standard Compustat sanity cut that drops duplicate FS-format rows and
restated views.

In [ ]:
SQL = """
SELECT  b.lpermno   AS permno,
        n.ticker    AS ticker,
        n.shrcd,
        n.exchcd,
        a.gvkey,
        a.rdq       AS announce_date,
        a.datadate  AS period_end,
        a.fyearq,
        a.fqtr
FROM    comp.fundq AS a
JOIN    crsp.ccmxpf_linktable AS b
  ON    a.gvkey = b.gvkey
 AND    a.datadate BETWEEN b.linkdt AND COALESCE(b.linkenddt, CURRENT_DATE)
 AND    b.linktype IN ('LU', 'LC')
 AND    b.linkprim IN ('P', 'C')
LEFT JOIN crsp.msenames AS n
  ON    n.permno = b.lpermno
 AND    a.rdq BETWEEN n.namedt AND n.nameendt
WHERE   a.rdq IS NOT NULL
  AND   a.rdq BETWEEN %(start)s AND %(end)s
  AND   a.indfmt  = 'INDL'
  AND   a.datafmt = 'STD'
  AND   a.consol  = 'C'
  AND   a.popsrc  = 'D'
"""

df = db.raw_sql(SQL, params={"start": START, "end": END},
                date_cols=["announce_date", "period_end"]).copy()
df["permno"] = df["permno"].astype("int32")
df["ticker"] = df["ticker"].astype("string").str.upper().str.strip()
df = (df.sort_values(["permno", "announce_date"])
        .drop_duplicates(subset=["permno", "announce_date"])
        .reset_index(drop=True))
print(f"tickers matched: {df['ticker'].notna().sum():,} / {len(df):,}")

print(f"rows:           {len(df):,}")
print(f"unique permnos: {df['permno'].nunique():,}")
print(f"date range:     {df['announce_date'].min().date()} -> {df['announce_date'].max().date()}")
df.head()

In [ ]:
# Save as CSV on WRDS scratch.
scratch_root = Path(os.environ.get("WRDS_SCRATCH", f"/scratch/eur/{os.environ.get('USER', '$USER')}"))
save_path = Path(os.environ.get("EARNINGS_OUT_DIR", scratch_root / "earnings_dates"))
save_path.mkdir(parents=True, exist_ok=True)
df.to_csv(save_path / "earnings_dates.csv", index=False)


## Write CSV

## Build the EarningsWindow indicator (on your laptop, after download)




In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == "wrds_cloud" else Path.cwd()
earnings_path = repo_root / "data" / "licensed" / "earnings_dates.csv"
panel_path = repo_root / "data" / "licensed" / "work_panel.parquet"
out_path = repo_root / "data" / "licensed" / "work_panel_with_earnings.parquet"

ea = pd.read_csv(earnings_path, usecols=["permno", "announce_date"])
ea["permno"] = ea["permno"].astype("int32")
ea["announce_date"] = pd.to_datetime(ea["announce_date"])

panel = pd.read_parquet(panel_path).copy()
panel["permno"] = panel["permno"].astype("int32")
panel["date"] = pd.to_datetime(panel["date"])

windows = (
    pd.concat(
        [ea.assign(date=ea["announce_date"] + pd.Timedelta(days=offset)) for offset in (-1, 0, 1)],
        ignore_index=True,
    )
    [["permno", "date"]]
    .drop_duplicates()
    .assign(earnings_window=1)
)

panel = panel.merge(windows, on=["permno", "date"], how="left")
panel["earnings_window"] = panel["earnings_window"].fillna(0).astype("int8")
panel.to_parquet(out_path, index=False)

print(f"earnings announcements: {len(ea):,}")
print(f"panel rows:             {len(panel):,}")
print(f"earnings-window rows:   {panel['earnings_window'].sum():,}")
print(f"saved:                  {out_path}")

For trading-day windows (skipping weekends), shift through your trading
calendar instead of using `Timedelta(days=d)`.